### Working with LLM SQL Queries in Databricks

![AI_Query_Flow](./Assets/ai_query_flow.png)

### Loading CSV file into Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS genai_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/genai_lab"

# List of files to download
files = ["restaurant_reviews.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/GenAI_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Creating the Bronze Layer Dataframe

In [0]:
bronze_df = spark.read.load(f'/Volumes/{catalog_name}/default/genai_lab/restaurant_reviews.csv', format='csv', header=True)
display(bronze_df.limit(10))

### Creating the Bronze Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Bronze;

### Storing the Bronze Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
bronzeDeltaTablePath = f'/Volumes/{catalog_name}/default/genai_lab/delta/bronze_restaurant_reviews'
bronze_df.write.format('delta').mode('overwrite').save(bronzeDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
bronze_df.write.format('delta').saveAsTable('Bronze.bronze_restaurant_reviews')

### Creating the Gold Layer by Performing Business Operations

### Loading the Gold Layer Dataframe from the Bronze Delta Table

In [0]:
gold_df = spark.read.format('delta').table('Bronze.bronze_restaurant_reviews')
display(gold_df.limit(10))

### Using LLM to Generate Sentiment Analysis and Summarization

In [0]:
gold_df = spark.sql("""
SELECT
    CustomerID,
    CustomerName,
    Products,
    Review,

    ai_query(
      "databricks-claude-sonnet-4-5",
      "Classify the customer sentiment as Positive, Neutral, or Negative. 
       Respond with only one word.: " || Review
    ) AS sentiment,

    ai_query(
      "databricks-claude-sonnet-4-5",
      "Summarize this restaurant review in one short sentence.: " || Review
    ) AS summary

FROM YOUR_UC_CATALOG_NAME.Bronze.bronze_restaurant_reviews
 """)

display(gold_df.limit(10))

### Creating the Gold Schema in Unity Catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Gold;

### Stroing the Gold Layer as a Delta Table in Unity Catalog

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
goldDeltaTablePath = f'/Volumes/{catalog_name}/default/genai_lab/delta/gold_restaurant_reviews'
gold_df.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
gold_df.write.format('delta').saveAsTable('Gold.restaurant_reviews')